# Task 02B — full decision-space signature study

Select an **NVIDIA GPU runtime** before starting; CPU is valid but may take hours. This notebook reruns fresh SAAS NUTS at the saved Task 02A checkpoints to obtain per-particle EI signatures. The cheap retrospective runs first. `RUN_FULL` is false, so Run all cannot launch the expensive extraction. Nothing here authenticates to or pushes to GitHub.

In [ ]:
REPO_URL = "https://github.com/PaulsonLab/energy-inference-bo.git"
REPO_REF = "772f59fb029bbebdff3bb22885988c1d37661b89"  # Reviewed full-run source.
ACCELERATOR = "gpu" # Choose "gpu" (preferred) or "cpu".
RUN_FULL = False       # Change only after backend validation, tests, and preflight pass.
PROFILE = "full"


In [ ]:
import os, platform, sys
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(f"Python {platform.python_version()} is unsupported; use Colab Python 3.11 or 3.12.")
if ACCELERATOR not in {"cpu", "gpu"}:
    raise ValueError("ACCELERATOR must be 'cpu' or 'gpu'.")
os.environ["JAX_PLATFORMS"] = "cuda" if ACCELERATOR == "gpu" else "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
print("Python", platform.python_version(), "requested accelerator", ACCELERATOR)


In [ ]:
from pathlib import Path
import subprocess
REPO_DIR = Path("/content/energy-inference-bo")
if REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} already exists. Restart the runtime for a clean archival run.")
if ACCELERATOR == "gpu":
    subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Checked out", GIT_SHA)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pytest==9.1.1"], check=True)
if ACCELERATOR == "gpu":
    subprocess.run([sys.executable, "-m", "pip", "install", "jax[cuda12]==0.9.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)


In [ ]:
import importlib.metadata as metadata
import jax
devices = jax.devices()
if ACCELERATOR == "gpu" and (jax.default_backend() != "gpu" or not any(device.platform == "gpu" for device in devices)):
    raise RuntimeError(f"GPU was requested but JAX reports backend={jax.default_backend()} devices={devices}")
if ACCELERATOR == "cpu" and jax.default_backend() != "cpu":
    raise RuntimeError(f"CPU was requested but JAX reports {jax.default_backend()}")
print("JAX", jax.__version__, jax.default_backend(), devices)
for package in ["energy-inference-bo", "torch", "botorch", "gpytorch", "numpyro"]:
    print(package, metadata.version(package))
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


In [ ]:
OUTPUT_DIR = REPO_DIR / "artifacts/task02b/full"
RETROSPECTIVE_DIR = OUTPUT_DIR / "retrospective"
TASK02A_RESULTS = REPO_DIR / "results/task02a/full"
RETROSPECTIVE_COMMAND = [sys.executable, "-m", "energy_bo.experiments.run_task02b", "--profile", "retrospective", "--task02a-results", str(TASK02A_RESULTS), "--output-dir", str(RETROSPECTIVE_DIR)]
subprocess.run(RETROSPECTIVE_COMMAND, cwd=REPO_DIR, check=True)
print("Retrospective preflight complete:", RETROSPECTIVE_DIR)


## Explicit full run

The next cell processes all 18 checkpoints sequentially with fresh NUTS. Set `RUN_FULL = True` only after tests and the retrospective preflight pass.

In [ ]:
if not RUN_FULL:
    raise RuntimeError("Full run is disabled. Review the configuration, set RUN_FULL=True, and rerun this cell.")
FULL_COMMAND = [sys.executable, "-m", "energy_bo.experiments.run_task02b", "--profile", PROFILE, "--task02a-results", str(TASK02A_RESULTS), "--output-dir", str(OUTPUT_DIR), "--retrospective-dir", str(RETROSPECTIVE_DIR), "--summary-path", str(OUTPUT_DIR / "SUMMARY.md")]
subprocess.run(FULL_COMMAND, cwd=REPO_DIR, check=True, env=os.environ.copy())


In [ ]:
import json, shutil
manifest = {
    "task": "task02b", "profile": PROFILE, "git_sha": GIT_SHA,
    "python": platform.python_version(), "accelerator": ACCELERATOR,
    "jax_backend": jax.default_backend(), "jax_devices": [str(device) for device in devices],
    "packages": {name: metadata.version(name) for name in ["jax", "jaxlib", "numpyro", "torch", "botorch", "gpytorch"]},
    "commands": [RETROSPECTIVE_COMMAND, FULL_COMMAND],
}
(OUTPUT_DIR / "colab_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
archive = Path(shutil.make_archive("/content/task02b_full_outputs", "zip", root_dir=REPO_DIR, base_dir="artifacts/task02b/full"))
print(archive, archive.stat().st_size, "bytes")
from google.colab import files
files.download(str(archive))


## What to do with the ZIP

Unzip locally and preserve `artifacts/task02b/full/` while reviewing. Compare it with the published `results/task02b/full/` package; do not overwrite evidence automatically. For a deliberately published revision, retain the summary, manifest, aggregate tables/configuration, import audit, checksum inventory, and only the plots used by the summary. Do **not** commit per-particle signature matrices; keep those in ignored artifacts or external storage. Commit and push manually.